# P053 — DRAM Memory Yield Predictor: Colab Simulation v5

**What this does:** Runs the full 40-day production simulation on Colab T4 GPU.

| Run | Rows/day | Retrain Epochs | Purpose | Est. Time |
|-----|----------|----------------|---------|----------------|
| fast | 100K | 5 | Sanity check | ~15 min |
| full | 5M | 50 | Production real T4 training | ~3-4 hours |

**Total: ~4-5 hours (within 7hr budget)**

Results: `MyDrive/P053_results/v5/{fast,full}/`

---

### How to run
1. Open on Google Colab
2. Runtime > Change runtime type > **T4 GPU**
3. Set Colab Secrets: `GITHUB_TOKEN`, optionally `AWS_ACCESS_KEY_ID` + `AWS_SECRET_ACCESS_KEY`
4. **Runtime > Run all**
5. Come back in ~5 hours, copy `v5/full/` to your machine

---
*P053 — AIML Engineering Lab*


In [ ]:
# ===============================================================
# CELL 1: Clone repo + install deps + verify code version
# ===============================================================
import subprocess, os
from pathlib import Path

PROJECT_DIR = '/content/053_memory_yield_predictor'
REPO = 'AIML-Engineering-Lab/053_dram_memory_yield_mlops'

if Path(PROJECT_DIR).exists():
    subprocess.run(['rm', '-rf', PROJECT_DIR], check=True)

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

clone_url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
result = subprocess.run(['git', 'clone', '--depth', '1', clone_url, PROJECT_DIR],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('CLONE FAILED. Add GITHUB_TOKEN to Colab Secrets (repo is private).')
    print(f'stderr: {result.stderr}')
    raise RuntimeError('git clone failed')
print(f'Cloned to {PROJECT_DIR}')

subprocess.run(['pip', 'install', '-q', '-r', f'{PROJECT_DIR}/requirements.txt'],
               check=True, capture_output=True, text=True)
print('Dependencies installed')

# Critical: verify --sim-retrain-epochs exists (was missing in v1 run)
verify = subprocess.run(['python', '-m', 'src.run_simulation', '--help'],
                        capture_output=True, text=True, cwd=PROJECT_DIR)
if '--sim-retrain-epochs' in verify.stdout:
    print('--sim-retrain-epochs flag: PRESENT')
else:
    print('ERROR: --sim-retrain-epochs NOT found! Push latest commits first.')
    raise RuntimeError('Code outdated on GitHub')


In [ ]:
# ===============================================================
# CELL 2: GPU verification + AWS credentials (optional)
# ===============================================================
import torch, os

if not torch.cuda.is_available():
    raise RuntimeError('No GPU! Runtime > Change runtime type > T4 GPU')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cc = torch.cuda.get_device_capability(0)
print(f'GPU: {gpu_name} ({vram_gb:.1f} GB, cc {cc[0]}.{cc[1]})')
print(f'AMP: {"bfloat16" if cc[0] >= 8 else "float16 + GradScaler"}')

# AWS credentials (optional, for S3 upload)
try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'
    print('AWS credentials: SET')
except Exception:
    print('AWS credentials: NOT SET (S3 skipped, Drive backup only)')


In [ ]:
# ===============================================================
# CELL 3: Mount Drive + copy preprocessed data to local SSD
# ===============================================================
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

PROJECT_DIR = '/content/053_memory_yield_predictor'
NPZ_DEST = Path(PROJECT_DIR) / 'data' / 'preprocessed_full.npz'
NPZ_DEST.parent.mkdir(parents=True, exist_ok=True)

candidates = [
    '/content/drive/MyDrive/P053_data/preprocessed_full.npz',
    '/content/drive/MyDrive/preprocessed_full.npz',
]
found = False
for src_path in candidates:
    if Path(src_path).exists():
        shutil.copy2(src_path, NPZ_DEST)
        size_gb = NPZ_DEST.stat().st_size / 1e9
        print(f'Copied preprocessed_full.npz ({size_gb:.2f} GB) from {src_path}')
        found = True
        break

if not found:
    print('WARNING: preprocessed_full.npz not found on Drive!')
    print('Retraining will be METADATA-ONLY (no real GPU training).')
    print(f'Upload to: {candidates[0]}')
else:
    print(f'Data ready for training')


In [ ]:
# ===============================================================
# CELL 4: Run simulations (fast + full) with Drive backup
# ===============================================================
#
# fast:  100K rows/day, 5 retrain epochs   (~15 min)
# full:  5M rows/day,  50 retrain epochs   (~3-4 hours)
#
# If Colab disconnects, re-run this cell. Checkpoint resumes.
# ===============================================================

import json, os, shutil, subprocess, sys, time
from pathlib import Path

PROJECT_DIR = '/content/053_memory_yield_predictor'
DRIVE_OUTPUT = '/content/drive/MyDrive/P053_results/v5'
Path(DRIVE_OUTPUT).mkdir(parents=True, exist_ok=True)

RUNS = [
    {'name': 'fast', 'rows': 100_000,   'epochs': 5},
    {'name': 'full', 'rows': 5_000_000, 'epochs': 50},
]

results_summary = []

for run_cfg in RUNS:
    run_name = run_cfg['name']
    rows = run_cfg['rows']
    epochs = run_cfg['epochs']

    print(f"\n{'=' * 70}")
    print(f'STARTING {run_name.upper()} | rows/day={rows:,} | retrain_epochs={epochs}')
    print(f"{'=' * 70}")

    # Clear checkpoint + timeline from previous mode
    for fname in ['simulation_checkpoint.json', 'simulation_timeline.json']:
        p = Path(PROJECT_DIR) / 'data' / fname
        p.unlink(missing_ok=True)

    cmd = [
        sys.executable, '-m', 'src.run_simulation',
        '--rows', str(rows),
        '--sim-retrain-epochs', str(epochs),
        '--backend', 'colab',
        '--checkpoint',
    ]
    print(f'CMD: {" ".join(cmd)}')

    env = os.environ.copy()
    t0 = time.time()

    result = subprocess.run(
        cmd, cwd=PROJECT_DIR, env=env,
        capture_output=True, text=True,
        timeout=25200,  # 7hr max
    )
    elapsed_min = (time.time() - t0) / 60

    # Print tail of stdout/stderr
    if result.stdout:
        print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.stderr:
        print('STDERR:', result.stderr[-2000:] if len(result.stderr) > 2000 else result.stderr)

    if result.returncode != 0:
        print(f'ERROR: {run_name} failed (exit {result.returncode}). Continuing...')
        results_summary.append({
            'name': run_name, 'rows': rows, 'epochs': epochs,
            'elapsed_min': elapsed_min, 'status': 'FAILED',
        })
        continue

    # ---- Backup to Drive ----
    backup_dir = Path(DRIVE_OUTPUT) / run_name
    backup_dir.mkdir(parents=True, exist_ok=True)

    for rel in ['data/simulation_timeline.json', 'data/simulation_checkpoint.json', 'mlflow.db']:
        src = Path(PROJECT_DIR) / rel
        if src.exists():
            dest_name = f'mlflow_{run_name}.db' if src.name == 'mlflow.db' else src.name
            shutil.copy2(src, backup_dir / dest_name)

    for f in (Path(PROJECT_DIR) / 'data').glob('benchmark_*.json'):
        shutil.copy2(f, backup_dir / f.name)

    for folder in ['drift_reports', 'production']:
        s = Path(PROJECT_DIR) / 'data' / folder
        d = backup_dir / folder
        if s.exists():
            if d.exists(): shutil.rmtree(d)
            shutil.copytree(s, d)

    for folder in ['mlruns', 'models']:
        s = Path(PROJECT_DIR) / folder
        d = backup_dir / folder
        if s.exists():
            if d.exists(): shutil.rmtree(d)
            shutil.copytree(s, d)

    # Read timeline summary
    tl_path = Path(PROJECT_DIR) / 'data' / 'simulation_timeline.json'
    retrains = 0
    total_days = 0
    retrain_info = ''
    if tl_path.exists():
        with open(tl_path) as f:
            tl = json.load(f)
        retrains = len(tl.get('retrain_events', []))
        total_days = tl.get('total_days', 0)
        for evt in tl.get('retrain_events', []):
            retrain_info += f' day{evt["day"]}={evt["new_model"]}'

    results_summary.append({
        'name': run_name, 'rows': rows, 'epochs': epochs,
        'elapsed_min': elapsed_min, 'retrains': retrains,
        'total_days': total_days, 'backup_dir': str(backup_dir),
        'status': 'OK', 'retrain_info': retrain_info.strip(),
    })

    print(f"\n{'='*50}")
    print(f'{run_name.upper()} COMPLETE')
    print(f'  Days: {total_days} | Retrains: {retrains}{retrain_info}')
    print(f'  Time: {elapsed_min:.1f} min ({elapsed_min/60:.1f} hr)')
    print(f'  Saved: {backup_dir}')
    print(f"{'='*50}")

# Save summary
with open(Path(DRIVE_OUTPUT) / 'run_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f'\nAll done. Summary: {DRIVE_OUTPUT}/run_summary.json')


In [ ]:
# ===============================================================
# CELL 5: Final summary
# ===============================================================
import json
from pathlib import Path

summary_path = Path('/content/drive/MyDrive/P053_results/v5/run_summary.json')
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print('=' * 70)
    print('ALL RUNS COMPLETE')
    print('=' * 70)
    total_min = 0.0
    for item in summary:
        total_min += item.get('elapsed_min', 0)
        print(
            f'  {item["name"]:<8} {item.get("status","?"):<6} '
            f'days={item.get("total_days","-"):<3} '
            f'retrains={item.get("retrains","-"):<2} '
            f'epochs={item.get("epochs","-"):<3} '
            f'time={item.get("elapsed_min",0):.1f} min'
        )
    print('-' * 70)
    print(f'TOTAL: {total_min:.1f} min ({total_min/60:.1f} hours)')
    print()
    print('Copy v5/full/ folder to your machine, then run:')
    print('  python -m src.post_simulation_update')
else:
    print('No summary found. Run Cell 4 first.')
